# 06 — Gradient Verification and Debugging (Section 9)

Covers analytical/numerical/autograd gradient comparison, finite differences, step size and precision effects, gradient accumulation bugs, and the Section 9 autograd cheat sheet.

**Authoritative cheat sheet source:** `notes/001_notes.md` → `## Autograd One-Page Cheat Sheet (Section 9)`. The diagram/content below is embedded as an identical copy for in-notebook requirement.


In [1]:
import numpy as np
import torch

torch.manual_seed(0)
np.random.seed(0)


## 1-5. Three functions: analytical derivatives, numerical finite differences, compare with autograd
## 6. Different finite-difference step sizes
## 7. Investigate floating-point precision effects


In [2]:
def numerical_grad(f, x_val, h):
  return (f(x_val + h) - f(x_val - h)) / (2.0 * h)

# Function 1: f(x)=x^3 - 2x + 1, f'(x)=3x^2 - 2
f1 = lambda x: x**3 - 2*x + 1
df1 = lambda x: 3*x**2 - 2
# Function 2: f(x)=sin(x) + x^2, f'(x)=cos(x) + 2x (use torch for autograd/numerical consistency)
f2_t = lambda x: torch.sin(x) + x**2
f2_n = lambda x: np.sin(x) + x**2
df2 = lambda x: np.cos(x) + 2*x
# Function 3: f(x)=exp(0.5x) + 1/x-ish safe: exp(0.5x) + x^4
f3_t = lambda x: torch.exp(0.5*x) + x**4
f3_n = lambda x: np.exp(0.5*x) + x**4
df3 = lambda x: 0.5*np.exp(0.5*x) + 4*x**3

print('Function 1: x^3 - 2x + 1 at x=1.5')
x = 1.5
analytical = df1(x)
numerical = numerical_grad(f1, x, 1e-5)
xt = torch.tensor(x, requires_grad=True)
yt = f1(xt); yt.backward()
autograd = xt.grad.item()
print('analytical:', analytical, 'numerical:', numerical, 'autograd:', autograd)
print('close analytical-numerical:', np.allclose(analytical, numerical, atol=1e-6, rtol=1e-6))
print('close analytical-autograd:', np.allclose(analytical, autograd, atol=1e-7, rtol=1e-7))

print('\nFunction 2: sin(x)+x^2 at x=0.7')
x = 0.7
analytical = df2(x)
numerical = numerical_grad(f2_n, x, 1e-5)
xt = torch.tensor(x, requires_grad=True)
yt = f2_t(xt); yt.backward()
autograd = xt.grad.item()
print('analytical:', analytical, 'numerical:', numerical, 'autograd:', autograd)

print('\nFunction 3: exp(0.5x)+x^4 at x=1.2')
x = 1.2
analytical = df3(x)
numerical = numerical_grad(f3_n, x, 1e-5)
xt = torch.tensor(x, requires_grad=True)
yt = f3_t(xt); yt.backward()
autograd = xt.grad.item()
print('analytical:', analytical, 'numerical:', numerical, 'autograd:', autograd)

print('\nStep size experiment (function 1 at x=1.5):')
for h in [1e-1, 1e-3, 1e-5, 1e-7, 1e-9]:
  num = numerical_grad(f1, 1.5, h)
  err = abs(num - df1(1.5))
  print('h={:<8} numerical={:.12f} abs_error={:.3e}'.format(h, num, err))

print('\nFloating-point precision experiment:')
for dtype, name in [(torch.float32, 'float32'), (torch.float64, 'float64')]:
  xt = torch.tensor(1.5, dtype=dtype, requires_grad=True)
  yt = xt**3 - 2*xt + 1
  yt.backward()
  ag = xt.grad.item()
  print(name, 'autograd grad:', ag, 'abs error vs analytical:', abs(ag - df1(1.5)))


Function 1: x^3 - 2x + 1 at x=1.5
analytical: 4.75 numerical: 4.750000000131038 autograd: 4.75
close analytical-numerical: True
close analytical-autograd: True

Function 2: sin(x)+x^2 at x=0.7
analytical: 2.1648421872844885 numerical: 2.164842187257765 autograd: 2.164842128753662

Function 3: exp(0.5x)+x^4 at x=1.2
analytical: 7.823059400195254 numerical: 7.823059400724474 autograd: 7.823060035705566

Step size experiment (function 1 at x=1.5):
h=0.1      numerical=4.760000000000 abs_error=1.000e-02
h=0.001    numerical=4.750000999999 abs_error=1.000e-06
h=1e-05    numerical=4.750000000131 abs_error=1.310e-10
h=1e-07    numerical=4.750000002218 abs_error=2.218e-09
h=1e-09    numerical=4.750000393017 abs_error=3.930e-07

Floating-point precision experiment:
float32 autograd grad: 4.75 abs error vs analytical: 0.0
float64 autograd grad: 4.75 abs error vs analytical: 0.0


## 8. Create an experiment where gradients are accidentally accumulated
## 9. Identify and correct the error

In [3]:
# Accidental accumulation bug
x = torch.tensor(2.0, requires_grad=True)
for i in range(3):
  y = x**2
  y.backward()
  print('Step', i, 'grad (accumulated):', x.grad.item())
print('Bug: gradient accumulated across iterations because grad was not cleared (4,8,12 expected progression).')

# Corrected
print('\nCorrected version:')
x = torch.tensor(2.0, requires_grad=True)
for i in range(3):
  if x.grad is not None:
    x.grad.zero_()
  y = x**2
  y.backward()
  print('Step', i, 'grad (cleared each step):', x.grad.item())


Step 0 grad (accumulated): 4.0
Step 1 grad (accumulated): 8.0
Step 2 grad (accumulated): 12.0
Bug: gradient accumulated across iterations because grad was not cleared (4,8,12 expected progression).

Corrected version:
Step 0 grad (cleared each step): 4.0
Step 1 grad (cleared each step): 4.0
Step 2 grad (cleared each step): 4.0


## 10. Autograd One-Page Cheat Sheet (Section 9)

**Authoritative source:** `notes/001_notes.md` → `## Autograd One-Page Cheat Sheet (Section 9)` (identical copy embedded below).

```mermaid
flowchart TD
  A[Create tensors] --> B{requires_grad?}
  B -->|Yes| C[Forward ops build graph]
  B -->|No| D[No grad history]
  C --> E[Scalar loss]
  E --> F[loss.backward]
  F --> G[.grad on leaf tensors]
  G --> H[Use gradients]
  H --> I{Need update?}
  I -->|Yes| J[torch.no_grad update]
  I -->|No| K[Inspect only]
  J --> L[Clear grads: grad.zero_ or None]
  L --> M[Next iteration]
  D --> N[torch.detach or no_grad blocks tracking]
  C --> N
  subgraph Debug
    O[Accumulation bug: missing zero_grad]
    P[Tracking bug: no_grad/detach/missing requires_grad]
    Q[Verify: analytical vs numerical vs autograd]
  end
```

**Quick rules**
- Track: set `requires_grad=True` on tensors whose gradients you need.
- Backward: call on scalar loss (or provide gradient argument for non-scalar outputs).
- Accumulation: gradients add up by default; clear before independent steps.
- Stop tracking: `torch.no_grad()` for blocks, `detach()` for tensors, `.item()` for Python scalars.
- Leaf/non-leaf: parameter/leaf grads are usually what you optimize.
- Verify: compare autograd with finite differences/analytical derivatives within tolerance.
